# 1. Project Introduction
Title: Cognifyz Task 1 - Restaurant Rating Prediction. ML Internship project.

# 2. Objective
Build a regression model to predict Aggregate rating.

# 3. Import Libraries
We import essential libraries for data manipulation, visualization, and machine learning using scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn import metrics


# 4. Load Dataset
Loading the data from the relative path.

In [ ]:
df = pd.read_csv('../data/restaurants.csv')
print("Dataset loaded successfully.")


# 5. Dataset Overview
Understanding the shape and features of our dataset.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nInfo:")
df.info()
display(df.head())


# 6. Data Quality Analysis
Summary statistics and outlier checks.

In [ ]:
display(df.describe())
# Simple boxplot to check for outliers in Aggregate rating
plt.figure(figsize=(8,4))
sns.boxplot(x=df['Aggregate rating'])
plt.title('Outlier Check: Aggregate Rating')
plt.show()


# 7. Missing Values
Checking for and visualizing missing data.

In [ ]:
missing = df.isnull().sum()
print("Missing values:\n", missing[missing > 0])
plt.figure(figsize=(10,5))
missing[missing > 0].plot(kind='bar')
plt.title('Missing Values Count')
plt.ylabel('Count')
plt.show()


# 8. Duplicate Analysis
Identifying any duplicate rows in the dataset.

In [ ]:
dup_count = df.duplicated().sum()
print(f"Number of duplicate rows: {dup_count}")


# 9. Target Variable Analysis
Distribution of 'Aggregate rating'. All 2148 zero ratings are 'Not rated' restaurants (Rating text == 'Not rated', Rating color == 'White'). They represent unrated restaurants, not actual zero scores.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Aggregate rating'], bins=30, kde=True)
plt.title('Distribution of Aggregate Rating')
plt.show()

zero_ratings = df[df['Aggregate rating'] == 0.0]
print("Count of zero ratings:", len(zero_ratings))
print("Rating text for zero ratings:\n", zero_ratings['Rating text'].value_counts())
print("Rating color for zero ratings:\n", zero_ratings['Rating color'].value_counts())


# 10. Exploratory Data Analysis
Visualizing relationships between features and the target variable.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='Votes', y='Aggregate rating', data=df)
plt.title('Votes vs Aggregate Rating')
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x='Has Table booking', y='Aggregate rating', data=df)
plt.title('Table Booking vs Aggregate Rating')
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x='Has Online delivery', y='Aggregate rating', data=df)
plt.title('Online Delivery vs Aggregate Rating')
plt.show()

print("Country Code value counts:\n", df['Country Code'].value_counts())

plt.figure(figsize=(8,5))
sns.countplot(x='Price range', data=df)
plt.title('Price Range Distribution')
plt.show()


# 11. Feature Selection
Documenting excluded and selected features.

Excluded features:
- Restaurant ID: identifier
- Restaurant Name: high cardinality (7446 unique)
- Address: high cardinality (8918 unique)
- Locality: high cardinality (1208 unique)
- Locality Verbose: high cardinality (1265 unique)
- Rating color: DATA LEAKAGE
- Rating text: DATA LEAKAGE
- Switch to order menu: constant (all 'No')

Selected features:
- Numerical: Longitude, Latitude, Average Cost for two, Votes, Price range
- Categorical: Country Code, City, Cuisines, Currency, Has Table booking, Has Online delivery, Is delivering now


In [ ]:
numeric_features = ['Longitude', 'Latitude', 'Average Cost for two', 'Votes', 'Price range']
categorical_features = ['Country Code', 'City', 'Cuisines', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now']
target = 'Aggregate rating'

df['Country Code'] = df['Country Code'].astype(str)
df['Cuisines'] = df['Cuisines'].fillna('Unknown')


# 12. Data Preprocessing
Setting up ColumnTransformer pipelines for numerics and categoricals.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


# 13. Train-Test Split
80/20 split, random_state=42.

In [ ]:
X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


# 14. Linear Regression
Training a basic Linear Regression model.

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("Linear Regression Metrics:")
print("MAE:", metrics.mean_absolute_error(y_test, y_pred_lr))
print("MSE:", metrics.mean_squared_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, y_pred_lr)))
print("R2 Score:", metrics.r2_score(y_test, y_pred_lr))


# 15. Decision Tree Regression
Training a non-linear Decision Tree model.

In [ ]:
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

print("Decision Tree Metrics:")
print("MAE:", metrics.mean_absolute_error(y_test, y_pred_dt))
print("MSE:", metrics.mean_squared_error(y_test, y_pred_dt))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, y_pred_dt)))
print("R2 Score:", metrics.r2_score(y_test, y_pred_dt))


# 16. Model Evaluation
Calculating evaluation metrics for both models.

In [ ]:
def evaluate_model(y_true, y_pred):
    return {
        'MAE': metrics.mean_absolute_error(y_true, y_pred),
        'MSE': metrics.mean_squared_error(y_true, y_pred),
        'RMSE': np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
        'R2': metrics.r2_score(y_true, y_pred)
    }

lr_metrics = evaluate_model(y_test, y_pred_lr)
dt_metrics = evaluate_model(y_test, y_pred_dt)


# 17. Model Comparison
Comparing the performance metrics in a DataFrame.

In [ ]:
comparison_df = pd.DataFrame([lr_metrics, dt_metrics], index=['Linear Regression', 'Decision Tree'])
display(comparison_df)


# 18. Feature Importance
Extracting top 15 features from the Decision Tree model. Note: This shows model-derived importance, not necessarily causation.

In [ ]:
feature_names = dt_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = dt_pipeline.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10,6))
feat_imp.plot(kind='bar')
plt.title('Top 15 Feature Importances (Decision Tree)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


# 19. Sample Prediction
Demonstrating the prediction function with sample dictionary input.

In [ ]:
def predict_rating(sample_dict):
    sample_df = pd.DataFrame([sample_dict])
    sample_df['Country Code'] = sample_df['Country Code'].astype(str)
    pred = dt_pipeline.predict(sample_df)
    return pred[0]

sample_input = {
    'Longitude': 77.2,
    'Latitude': 28.6,
    'Average Cost for two': 500,
    'Votes': 100,
    'Price range': 2,
    'Country Code': '1',
    'City': 'New Delhi',
    'Cuisines': 'North Indian',
    'Currency': 'Indian Rupees(Rs.)',
    'Has Table booking': 'No',
    'Has Online delivery': 'Yes',
    'Is delivering now': 'No'
}

print("Predicted Rating:", predict_rating(sample_input))


# 20. Conclusion
Based on the evaluation metrics, the Decision Tree significantly outperforms the Linear Regression model, effectively capturing non-linear relationships in the data. The features like Votes and Average Cost play a vital role in determining a restaurant's aggregate rating.